### **Earthquake Event Data - Silver Layer**

**Below Step is to connect ADLS with **Databricks****

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

In [0]:
dbutils.widgets.text("file_name", "")
file_name = dbutils.widgets.get("file_name")
df = spark.read.option("multiline","true").json(f"abfss://dummydata@adlsweatherapidata.dfs.core.windows.net/API_Data/{file_name}")



In [0]:
from pyspark.sql.functions import explode

df = \
df.select(explode('features').alias('feature')).select('feature.*').\
    select(
           'id',
           col('geometry.coordinates').getItem(0).alias('Longitude'),
           col('geometry.coordinates').getItem(1).alias('Latitude'),
           col('geometry.coordinates').getItem(2).alias('elevation'),
           col('properties.title').alias('title'),
           col('properties.place').alias('place'),
           col('properties.sig').alias('sig'),
           col('properties.mag').alias('mag'),
           col('properties.magType').alias('magType'),
           col('properties.time').alias('time'),
           col('properties.updated').alias('updated')
    )

In [0]:
df = df.\
    withColumn('time',col('time')/1000).\
    withColumn('updated', col('updated')/1000).\
    withColumn('time',col('time').cast(TimestampType())).\
    withColumn('updated',col('updated').cast(TimestampType()))

In [0]:
from datetime import datetime

current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
file_name = f"earthquake_data_{current_time}.json"

df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(f"abfss://dummydata@adlsweatherapidata.dfs.core.windows.net/weather/silver/{file_name}")

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("earthquake_silver")